# Generate-then-Refine en línea (R6) — generar con Claude dentro de cada fold y evaluar

Claude (`claude-sonnet-4-6`) genera oraciones nuevas por categoría **dentro del entrenamiento de cada fold**, y se evalúa con la validación
cruzada de 5 folds. Todo lo hace `validacion_cruzada_en_linea.py`.

**Estrategia (la misma que el entrenamiento): test / pool / core / dev.** Para cada fold: *test* = el fold (no se toca);
*pool* = los otros 4 folds; el pool se parte en *dev interno* (~1/6) y *core*. Hay dos etapas:
- **Etapa core (elegir `C`):** el aumento se genera/filtra solo con el core y se mide en el dev interno.
- **Etapa pool (modelo final):** el aumento se genera/filtra con todo el pool y se mide en el test.

Así, el dev interno nunca influye en el aumento con el que se elige `C`. En todas las etapas se descartan las filas sintéticas
idénticas a una oración del test.

En Generate-then-Refine esto significa **dos generaciones por fold**: una usando solo el core (ejemplos few-shot y centroide del filtro
semántico) para la etapa de elegir `C`, y otra usando todo el pool para el modelo final. (La evaluación vigente genera una sola vez con el
pool y la usa en las dos etapas; aquí el dev interno ya no influye en el aumento con el que se elige `C`.)

**Antes de correr:**
1. `git commit` + `git push` del repo reorganizado (carpetas `oe2_...`) con `validacion_cruzada_en_linea.py`. Esta Colab clona tu repo.
2. En Colab: ícono de llave (**Secretos**) → agrega `ANTHROPIC_API_KEY` con tu clave y activa "Acceso desde el cuaderno".
3. No hace falta GPU (con GPU, los embeddings salen más rápido).

**Costo en tokens (estimación, verifícala en tu consola de Anthropic):** 5 folds × 2 etapas × 7 categorías = **70 llamadas** a la API; cada una pide
20 pares español-shiwilu (~1-2 mil tokens de salida y ~1.5 mil de entrada). En total ~80-140 mil tokens de salida: del orden de **1-2 dólares**.
Lo generado se guarda en Drive apenas se produce (`shiwilu_en_linea/`): si algo falla a medias y vuelves a correr el paso 4, **no se vuelve a pagar**
lo ya generado.

## 0. Montar Google Drive (aquí queda guardado lo generado)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Clonar tu repo

In [ ]:
import os
GITHUB_USUARIO = "LhiaHC"

%cd /content
!rm -rf /content/shiwilu-tesis
!git clone https://github.com/{GITHUB_USUARIO}/shiwilu-tesis.git
%cd /content/shiwilu-tesis
!git log --oneline -3

ruta = '/content/shiwilu-tesis/oe2_aumento_de_datos/evaluacion/validacion_cruzada_en_linea.py'
assert os.path.exists(ruta), ('No existe validacion_cruzada_en_linea.py: haz git commit + git push del repo reorganizado '
                              '(carpetas oe2_...) con los cambios nuevos y vuelve a correr esta celda.')
print('OK')

## 2. Instalar dependencias

In [ ]:
!pip install -q anthropic python-dotenv sentence-transformers scikit-learn
import torch
print('GPU disponible:', torch.cuda.is_available())

## 3. API key (desde los Secretos de Colab) y prueba de conexión

La clave nunca se imprime. La prueba usa 10 tokens de salida.

In [ ]:
import os
from google.colab import userdata
import anthropic

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
cliente = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
r = cliente.messages.create(model="claude-sonnet-4-6", max_tokens=10, messages=[{"role": "user", "content": "Responde solo: ok"}])
print("Conexion OK:", r.content[0].text)

## 4. Generar y evaluar, fold por fold (70 llamadas a Claude)

Para cada fold: calcula test / pool / core / dev; genera con el core (`fold<N>_core.csv`) y con el pool (`fold<N>_pool.csv`), guardando cada
archivo en Drive; se quedan solo las filas aprobadas por los filtros y que no copian el test; extrae embeddings de los 3 modelos y entrena.
Para generar otra cantidad por categoría y etapa cambia `CANTIDAD` (el repo usó 20).

In [ ]:
import os, subprocess

def correr(comando, cwd="/content/shiwilu-tesis"):
    """Corre un comando mostrando la salida en vivo; se detiene si falla."""
    proc = subprocess.Popen(
        comando, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        env={**os.environ, "MPLBACKEND": "Agg", "TOKENIZERS_PARALLELISM": "false"},
    )
    for linea in proc.stdout:
        print(linea, end="")
    proc.wait()
    assert proc.returncode == 0, f"Fallo (codigo {proc.returncode}): {' '.join(comando)}"

PRUEBA = False   # True = prueba rapida y barata: solo el fold 0 (14 llamadas), con cache y etiqueta aparte
CANTIDAD = 20
CACHE = "/content/drive/MyDrive/shiwilu_en_linea" + ("_prueba" if PRUEBA else "")
ETQ = "colab" + ("_prueba" if PRUEBA else "")

correr(["python", "oe2_aumento_de_datos/evaluacion/validacion_cruzada_en_linea.py",
        "--tecnica", "generate_then_refine", "--cantidad", str(CANTIDAD), "--cache", CACHE, "--etiqueta", ETQ]
       + (["--folds", "0"] if PRUEBA else []))

## 5. Revisar lo generado (tasas de aprobación por fold y etapa)

In [ ]:
import glob
import pandas as pd
from IPython.display import display

filas = []
for ruta in sorted(glob.glob(f"{CACHE}/generate_then_refine/fold*_*.csv")):
    g = pd.read_csv(ruta)
    aprobadas = g[g.estado_filtro == "aprobado"].intencion.value_counts()
    fila = {"archivo": os.path.basename(ruta), "generadas": len(g), "aprobadas": int(aprobadas.sum())}
    fila.update({c: int(aprobadas.get(c, 0)) for c in ["AFI", "DES", "EMO", "NEG", "PRG", "REQUEST", "SAL"]})
    filas.append(fila)
display(pd.DataFrame(filas))
display(g[g.estado_filtro == "aprobado"].sample(8, random_state=0)[["espanol", "shiwilu", "intencion", "similitud_labse"]])

## 6. Tablas y curvas ROC (las mismas que el repo)

El CSV de predicciones de esta corrida trae **los 12 experimentos** (Generate-then-Refine nuevo + las otras tres configuraciones vigentes) con las
probabilidades por categoría, así que se pasa directo a `comparacion_pareada.py` y `curvas_roc.py`. Se generan: `comparacion_pareada_sin_puntuacion_<TAG>.csv`,
`curvas_roc_<TAG>.csv`, `curvas_roc_<TAG>.png` (grilla 3 × 4) y la carpeta `curvas_roc_<TAG>/` con una imagen por experimento.

In [ ]:
if not PRUEBA:
    import pandas as pd
    from IPython.display import display
    RES = "/content/shiwilu-tesis/oe2_aumento_de_datos/evaluacion/resultados"
    TAG = f"en_linea_generate_then_refine_{ETQ}"
    ENTRADA = f"{RES}/validacion_cruzada_predicciones_sin_puntuacion_{TAG}.csv"

    # mismas tablas y figuras que el repo: comparaciones pareadas, curvas ROC (CSV + grilla + una imagen por experimento)
    correr(["python", "oe2_aumento_de_datos/evaluacion/comparacion_pareada.py", "--entrada", ENTRADA, "--etiqueta", TAG])
    correr(["python", "oe2_aumento_de_datos/evaluacion/curvas_roc.py", "--entrada", ENTRADA, "--etiqueta", TAG])

    from IPython.display import Image
    display(Image(f"{RES}/curvas_roc_{TAG}.png"))
    auc_macro = pd.read_csv(f"{RES}/curvas_roc_{TAG}.csv").query("categoria == 'macro'")
    display(auc_macro.pivot(index="modelo", columns="tecnica", values="auc").round(4))

if PRUEBA:
    print('Modo prueba: solo se evaluo el fold 0; las tablas y curvas ROC necesitan los 5 folds.')

## 7. Comparar con `sin_aumento` y con Generate-then-Refine vigente (se generó una vez por fold con el pool)

In [ ]:
import sys
import numpy as np
import pandas as pd
from IPython.display import display

sys.path.insert(0, "/content/shiwilu-tesis/oe2_aumento_de_datos/evaluacion")
from comparacion_pareada import diferencia_pareada

RES = "/content/shiwilu-tesis/oe2_aumento_de_datos/evaluacion/resultados"
vigente = pd.read_csv(f"{RES}/validacion_cruzada_resumen_sin_puntuacion.csv")
P_vig = pd.read_csv(f"{RES}/validacion_cruzada_predicciones_sin_puntuacion.csv")

def f1(tabla, tecnica):
    return tabla[tabla.tecnica == tecnica].set_index("modelo").f1_macro_agrupado
if not PRUEBA:
    nuevo = pd.read_csv(f"{RES}/validacion_cruzada_resumen_sin_puntuacion_en_linea_generate_then_refine_{ETQ}.csv")
    display(pd.DataFrame({
        "sin aumento (vigente)": f1(vigente, "sin_aumento"),
        "GtR vigente": f1(vigente, "generate_then_refine"),
        "GtR EN LINEA (esta corrida)": f1(nuevo, "generate_then_refine"),
    }).round(4))

    P_nuevo = pd.read_csv(f"{RES}/validacion_cruzada_predicciones_sin_puntuacion_en_linea_generate_then_refine_{ETQ}.csv")
    P = pd.concat([
        P_nuevo[P_nuevo.tecnica == "generate_then_refine"].assign(tecnica="gtr_en_linea"),
        P_vig[P_vig.tecnica == "sin_aumento"],
        P_vig[P_vig.tecnica == "generate_then_refine"].assign(tecnica="gtr_vigente"),
    ])
    rng = np.random.default_rng(42)
    filas = []
    for modelo in ["labse", "mbert", "xlmr"]:
        for a, b in [("gtr_en_linea", "sin_aumento"), ("gtr_en_linea", "gtr_vigente")]:
            d, lo, hi = diferencia_pareada(P, (modelo, a), (modelo, b), rng)
            filas.append({"modelo": modelo, "comparacion": f"{a} - {b}", "diferencia_f1": round(d, 4),
                          "ic95_bajo": round(lo, 4), "ic95_alto": round(hi, 4), "distinguible_de_cero": not (lo <= 0 <= hi)})
    display(pd.DataFrame(filas))

if PRUEBA:
    print('Modo prueba: solo se evaluo el fold 0; la comparacion pareada necesita los 5 folds.')

## 8. Descargar el resultado

Un `.zip` con lo generado (`generate_then_refine/fold*_{core,pool}.csv`) y los resultados: predicciones y resumen (12 experimentos), comparaciones pareadas,
curvas ROC (CSV, grilla e imágenes por experimento). En tu computadora: lo generado va en `oe2_aumento_de_datos/tecnicas_aumento/salidas/en_linea/` y todo lo demás en
`oe2_aumento_de_datos/evaluacion/resultados/`. Lo generado también queda en Drive (`shiwilu_en_linea/`).

In [ ]:
TAG = f"en_linea_generate_then_refine_{ETQ}"

import glob, shutil
from google.colab import files

RES = "/content/shiwilu-tesis/oe2_aumento_de_datos/evaluacion/resultados"
carpeta = "/content/salida_gtr_en_linea"
shutil.rmtree(carpeta, ignore_errors=True)
os.makedirs(carpeta + "/resultados")
os.makedirs(carpeta + "/en_linea")
shutil.copytree(f"{CACHE}/generate_then_refine", carpeta + "/en_linea/generate_then_refine")
for ruta_ in glob.glob(f"{RES}/*{TAG}*"):      # CSV, PNG y la carpeta con una imagen por experimento
    destino = carpeta + "/resultados/" + os.path.basename(ruta_)
    shutil.copytree(ruta_, destino) if os.path.isdir(ruta_) else shutil.copy(ruta_, destino)
zip_path = shutil.make_archive("/content/salida_gtr_en_linea", "zip", carpeta)
print(sorted(os.listdir(carpeta + "/resultados")))
files.download(zip_path)